# 📘 01_generate_sample_data.ipynb

## 1. Introduction
This notebook generates **synthetic insurance-like sample data** for Lakehouse Monitoring demos.  
The same data will be used consistently across all blog series versions (`v0.1`, `v0.2`, ...).

We generate three Delta tables:
- **policies** → Insurance policies (start/end dates, premiums, status).
- **claims** → Insurance claims linked to policies (timestamps, amounts, status).
- **premium_billing** → Billing records for policies (due dates, amounts, invoice status).

## 2. Notebook Widgets
We use widgets to parameterize the execution:
- `catalog` → Unity Catalog catalog for table creation.
- `data_schema` → Schema to store generated tables.
- `start_date` / `end_date` → Date range for generating data.

📌 *Tip: Change these values to regenerate data for any desired date window.*

In [0]:
# ───────── Widgets ─────────
dbutils.widgets.text("catalog",        "dbdemos_steventan",                 "Catalog")
dbutils.widgets.text("data_schema",    "lakehouse_monitoring",              "Data Schema")
dbutils.widgets.text("start_date",     "2025-09-01",                        "Start date (yyyy-MM-dd)")
dbutils.widgets.text("end_date",       "2025-09-30",                        "End date (yyyy-MM-dd)")

catalog     = dbutils.widgets.get("catalog").strip()
data_schema = dbutils.widgets.get("data_schema").strip()
start_date  = dbutils.widgets.get("start_date").strip()
end_date    = dbutils.widgets.get("end_date").strip()

In [0]:
from pyspark.sql import functions as F

# Basic validation
if start_date > end_date:
    raise ValueError(f"start_date ({start_date}) must be <= end_date ({end_date})")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{data_schema}")

# ---- Tables (types match your existing ones) ----
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{data_schema}.policies (
  policy_no       STRING  NOT NULL,
  customer_id     STRING  NOT NULL,
  agent_id        STRING,
  product_code    STRING  NOT NULL,
  status          STRING,
  start_date      DATE,
  end_date        DATE,
  premium_annual  DECIMAL(12,2),
  created_at      TIMESTAMP
) USING DELTA
""")
spark.sql(f"ALTER TABLE {catalog}.{data_schema}.policies SET TBLPROPERTIES ('delta.enableChangeDataFeed'='true')")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{data_schema}.claims (
  claim_id        STRING NOT NULL,
  policy_no       STRING NOT NULL,
  incident_id     STRING,
  claim_status    STRING,
  claim_amount    DECIMAL(12,2),
  reported_at     TIMESTAMP,
  closed_at       TIMESTAMP,
  reported_at_str STRING
) USING DELTA
""")
spark.sql(f"ALTER TABLE {catalog}.{data_schema}.claims SET TBLPROPERTIES ('delta.enableChangeDataFeed'='true')")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{data_schema}.premium_billing (
  invoice_id    STRING NOT NULL,
  policy_no     STRING NOT NULL,
  due_date      DATE,
  amount_due    DECIMAL(12,2),
  status        STRING,
  generated_at  TIMESTAMP
) USING DELTA
""")
spark.sql(f"ALTER TABLE {catalog}.{data_schema}.premium_billing SET TBLPROPERTIES ('delta.enableChangeDataFeed'='true')")

## 3. Data Characteristics
For each day in the chosen range:
- Generate ~100 "good" rows per table.
- Inject **5–30 "bad" rows per day** to simulate quality issues:
  - *policies*: invalid end dates before start dates.
  - *claims*: future timestamps, invalid claim statuses.
  - *premium_billing*: negative amounts.

This guarantees that Lakehouse Monitoring will detect non-trivial data quality problems.

In [0]:
from pyspark.sql import functions as F

def pad4(col):
    return F.lpad(F.col(col).cast("string"), 4, "0")

def mk_ts_on_day(col_day, seed):
    # Deterministic random-ish time per day
    h = (F.floor(F.rand(seed) * 24)).cast("int")
    m = (F.floor(F.rand(seed + 1) * 60)).cast("int")
    s = (F.floor(F.rand(seed + 2) * 60)).cast("int")
    return F.to_timestamp(
        F.concat_ws(" ",
            F.date_format(col_day, "yyyy-MM-dd"),
            F.format_string("%02d:%02d:%02d", h, m, s)
        )
    )

# Helpers that build policy_no using columns present in the DF
def pol_for_n():
    # uses columns yyyyMMdd + n
    return F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"),
                    F.lpad((F.col("n") % 100).cast("string"), 4, "0"))

def pol_for_b():
    # uses columns yyyyMMdd + b
    return F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"),
                    F.lpad((F.col("b") % 100).cast("string"), 4, "0"))

In [0]:
# ---------------- Dates & bad-counts ----------------
dates = (
    spark.range(1)
         .select(F.explode(F.sequence(F.to_date(F.lit(start_date)),
                                      F.to_date(F.lit(end_date)))).alias("d"))
         .withColumn("yyyyMMdd", F.date_format("d", "yyyyMMdd"))
)

# random bad counts per day in [5..30]
bad_counts = dates.select(
    "d", "yyyyMMdd",
    (F.abs(F.hash("d") + F.lit(1337)) % F.lit(26) + F.lit(5)).alias("bad_cnt")
)
bad_rows = (
    bad_counts
      .withColumn("idx_arr", F.expr("sequence(0, bad_cnt-1)"))
      .withColumn("b", F.explode("idx_arr"))
      .drop("idx_arr")
)

## 4. Idempotent Writes
Before inserting new rows, the notebook **removes any existing rows** within the selected date range.  
This makes the notebook **safe to re-run** without duplicating data.

In [0]:
# ───────── Good policies ─────────
good_policies = (
  dates.crossJoin(spark.range(100).toDF("n"))
       .withColumn("policy_no", F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"), pad4("n")))
       .withColumn("customer_id", F.concat(F.lit("CUST"), F.lpad((F.col("n")+1).cast("string"), 3, "0")))
       .withColumn("agent_id", F.concat(F.lit("AG"), F.lpad(((F.col("n") % 20) + 1).cast("string"), 3, "0")))
       .withColumn("product_code", F.expr("CASE WHEN n%3=0 THEN 'AUTO' WHEN n%3=1 THEN 'HEALTH' ELSE 'HOME' END"))
       .withColumn("status", F.expr("CASE WHEN n%10=0 THEN 'LAPSED' WHEN n%15=0 THEN 'CANCELLED' ELSE 'ACTIVE' END"))
       .withColumn("start_date", F.expr("date_sub(d, CAST(n % 60 AS INT))"))
       .withColumn("end_date",   F.expr("date_add(date_sub(d, CAST(n % 60 AS INT)), 30 + CAST(n % 365 AS INT))"))
       .withColumn("premium_annual", (F.lit(800) + (F.col("n") % 700)).cast("decimal(12,2)"))
       .withColumn("created_at", mk_ts_on_day(F.col("d"), seed=11))
       .select("policy_no","customer_id","agent_id","product_code","status",
               "start_date","end_date","premium_annual","created_at")
)

# ───────── Bad policies ─────────
bad_policies = (
  bad_rows
    .withColumn("policy_no", F.concat(F.lit("POLBAD-"), F.col("yyyyMMdd"), F.lit("-"), F.lpad(F.col("b").cast("string"), 4, "0")))
    .withColumn("customer_id", F.concat(F.lit("CUSTBAD"), F.lpad((F.col("b")+1).cast("string"), 3, "0")))
    .withColumn("agent_id", F.lit("AG999"))
    .withColumn("product_code", F.lit("AUTO"))
    .withColumn("status", F.lit("ACTIVE"))
    .withColumn("start_date", F.col("d"))
    .withColumn("end_date",   F.expr("date_sub(d, 1 + CAST(b % 5 AS INT))"))  # bad: end < start
    .withColumn("premium_annual", F.lit(1200).cast("decimal(12,2)"))
    .withColumn("created_at", mk_ts_on_day(F.col("d"), seed=13))
    .select("policy_no","customer_id","agent_id","product_code","status",
            "start_date","end_date","premium_annual","created_at")
)

policies_df = good_policies.unionByName(bad_policies)

# ───────── Idempotent delete/write ─────────
spark.sql(f"""
  DELETE FROM {catalog}.{data_schema}.policies
  WHERE created_at >= timestamp('{start_date}')
    AND created_at <  timestamp(date_add('{end_date}', 1))
""")

(policies_df
 .write.mode("append")
 .saveAsTable(f"{catalog}.{data_schema}.policies"))

print(f"✅ Seeded policies {start_date}..{end_date}: {policies_df.count()} rows written")

In [0]:
# ───────── Helpers for claims ─────────
def policy_for_n():
    return F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"), pad4("n"))

def policy_for_b():
    return F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"), F.lpad((F.col("b") % 100).cast("string"), 4, "0"))

# ───────── Good claims (100/day) ─────────
good_claims = (
  dates.crossJoin(spark.range(100).toDF("n"))
       .withColumn("claim_id",    F.concat(F.lit("CLM-"), F.col("yyyyMMdd"), F.lit("-"), pad4("n")))
       .withColumn("policy_no",   policy_for_n())
       .withColumn("incident_id", F.concat(F.lit("INC-"), F.col("yyyyMMdd"), F.lit("-"), pad4("n")))
       .withColumn("claim_status",
                   F.expr("CASE WHEN n%5=0 THEN 'CLOSED' WHEN n%7=0 THEN 'DENIED' ELSE 'OPEN' END"))
       .withColumn("claim_amount", (F.lit(500) + (F.col("n") % 5000)).cast("decimal(12,2)"))
       .withColumn("reported_at", mk_ts_on_day(F.col("d"), seed=21))
       .withColumn("closed_at",
                   F.when(F.col("claim_status")=="CLOSED", F.expr("reported_at + INTERVAL 1 DAY"))
                    .otherwise(F.lit(None).cast("timestamp")))
       .withColumn("reported_at_str", F.date_format(F.col("reported_at"), "yyyy-MM-dd"))
       .select("claim_id","policy_no","incident_id","claim_status","claim_amount","reported_at","closed_at","reported_at_str")
)

# ───────── Bad claims (5–30/day) ─────────
bad_claims = (
  bad_rows
    .withColumn("claim_id",    F.concat(F.lit("CLMBAD-"), F.col("yyyyMMdd"), F.lit("-"), F.lpad(F.col("b").cast("string"), 4, "0")))
    .withColumn("policy_no",   policy_for_b())
    .withColumn("incident_id", F.concat(F.lit("INCBAD-"), F.col("yyyyMMdd"), F.lit("-"), F.lpad(F.col("b").cast("string"), 4, "0")))
    .withColumn("claim_status", F.expr("CASE WHEN b%2=0 THEN 'REVIEW' ELSE 'PENDING' END"))  # unexpected
    .withColumn("claim_amount", (F.lit(300) + (F.col("b") % 700)).cast("decimal(12,2)"))
    .withColumn("reported_at",
        F.when((F.col("b") % 2) == 0,
               F.to_timestamp(F.concat_ws(' ', F.date_format(F.date_add(F.col("d"), 365), 'yyyy-MM-dd'), F.lit('05:05:05'))))
         .otherwise(mk_ts_on_day(F.col("d"), seed=22))
    )
    .withColumn("closed_at", F.lit(None).cast("timestamp"))
    .withColumn("reported_at_str",
        F.when(F.col("b") % 7 == 0, F.lit("2025-09-31"))
         .when(F.col("b") % 7 == 1, F.lit("2025/09/16"))
         .when(F.col("b") % 7 == 2, F.lit("20250916abc"))
         .otherwise(F.date_format(F.col("reported_at"), "yyyy-MM-dd"))
    )
    .select("claim_id","policy_no","incident_id","claim_status","claim_amount","reported_at","closed_at","reported_at_str")
)

claims_df = good_claims.unionByName(bad_claims)

# ───────── Idempotent load ─────────
spark.sql(f"""
  DELETE FROM {catalog}.{data_schema}.claims
  WHERE reported_at >= timestamp('{start_date}')
    AND reported_at <  timestamp(date_add('{end_date}', 1))
""")

(claims_df.write.mode("append")
  .saveAsTable(f"{catalog}.{data_schema}.claims"))

print(f"✅ Seeded claims {start_date}..{end_date}: {claims_df.count()} rows written")

In [0]:
from pyspark.sql import functions as F

# Tiny helpers for policy key pattern (reuses yyyyMMdd from `dates` / `bad_rows`)
def policy_for_n():
    return F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"), pad4("n"))

def policy_for_b():
    return F.concat(F.lit("POL-"), F.col("yyyyMMdd"), F.lit("-"), F.lpad((F.col("b") % 100).cast("string"), 4, "0"))

# ───────── Good invoices (100/day) ─────────
good_inv = (
  dates.crossJoin(spark.range(100).toDF("n"))
       .withColumn("invoice_id",  F.concat(F.lit("INV-"), F.col("yyyyMMdd"), F.lit("-"), pad4("n")))
       .withColumn("policy_no",   policy_for_n())
       .withColumn("due_date",    F.expr("date_add(d, 15)"))
       .withColumn("amount_due",  (F.lit(50) + (F.col("n") % 200)).cast("decimal(12,2)"))
       .withColumn("status",      F.expr("CASE WHEN n%6=0 THEN 'PAID' WHEN n%8=0 THEN 'OVERDUE' ELSE 'DUE' END"))
       .withColumn("generated_at", mk_ts_on_day(F.col("d"), seed=31))
       .select("invoice_id","policy_no","due_date","amount_due","status","generated_at")
)

# ───────── Bad invoices (5–30/day): negative amounts ─────────
bad_inv = (
  bad_rows
    .withColumn("invoice_id",  F.concat(F.lit("INVBAD-"), F.col("yyyyMMdd"), F.lit("-"), F.lpad(F.col("b").cast("string"), 4, "0")))
    .withColumn("policy_no",   policy_for_b())
    .withColumn("due_date",    F.expr("date_add(d, 10)"))
    .withColumn("amount_due",  (F.lit(-1) * (F.lit(20) + (F.col("b") % 50))).cast("decimal(12,2)"))  # negative
    .withColumn("status",      F.lit("DUE"))
    .withColumn("generated_at", mk_ts_on_day(F.col("d"), seed=33))
    .select("invoice_id","policy_no","due_date","amount_due","status","generated_at")
)

billing_df = good_inv.unionByName(bad_inv)

# ───────── Idempotent load for the chosen range ─────────
spark.sql(f"""
  DELETE FROM {catalog}.{data_schema}.premium_billing
  WHERE generated_at >= timestamp('{start_date}')
    AND generated_at <  timestamp(date_add('{end_date}', 1))
""")

(billing_df.write.mode("append")
  .saveAsTable(f"{catalog}.{data_schema}.premium_billing"))

print(f"✅ Seeded premium_billing {start_date}..{end_date}: {billing_df.count()} rows written")

## 5. Outputs
After running, the following tables are created/updated:
- `{catalog}.{data_schema}.policies`
- `{catalog}.{data_schema}.claims`
- `{catalog}.{data_schema}.premium_billing`

These tables will be consumed by:
- **02_metadata_tables.ipynb** → Register tables for monitoring.
- **03_run_monitoring.ipynb** → Run Lakehouse Monitoring jobs.